# Training: Channel Specificity (Fz, Pz, Oz)

**Scope:**

- **Subjects:** VALID_SUBJECTS, [1, 2, 4, 6, 7, 8, 9, 10, 12, 13, 15, 16, 17, 18, 19, 20, 21, 22]
- **Channels:** Pz, Oz  (Fz analyzed separately as the reference)
- **Time window:** 2 s
- **CV:** GroupKFold, 5 folds over 40 trials
- **Train overlap:** 50% | **Test overlap:** 0%

**Settings:**

- 10 epochs, batch size 16
- Adam: lr = 1e-4, weight_decay = 1e-5
- LR decayed by 0.1 every 5 epochs
- Same network and training protocol for all channels

**Purpose:**

Tests whether the video model's prediction performance is spatially
specific to the frontal channel (Fz) or generalizes to posterior
channels (Pz, Oz). Addresses Reviewer 1, Comment 1.

**Runs:**

- Pz — exp start 300
- Oz — exp start 400
- One run per (subject, channel)

**Output directory:** `Channel-Specificity/Results/`

**Model directory:** `Channel-Specificity/Models/`

In [ ]:
# ============================================================
# 0. Environment setup (Colab or local)
# ============================================================
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    try:
        get_ipython().run_line_magic('matplotlib', 'inline')
    except Exception:
        pass
    !pip install -q pims


# ============================================================
# 1. Standard library
# ============================================================
import os
import sys
import pickle
import traceback

# ============================================================
# 2. Third-party
# ============================================================
import numpy as np
import torch
import torch.nn as nn

# ============================================================
# 3. Project paths
# ============================================================
if IN_COLAB:
    SOURCE_ROOT = '/content/drive/MyDrive/Colab Notebooks/Face2EEG/Source'
    DATA_ROOT   = '/content/drive/MyDrive/Colab Notebooks/Face2EEG/Data'
else:
    SOURCE_ROOT = r'D:\Thesis\Codes\Source'
    DATA_ROOT   = r'D:\Thesis\Codes\Data'

if not os.path.isdir(os.path.join(SOURCE_ROOT, 'face2eeg')):
    raise FileNotFoundError(
        f"face2eeg package not found under SOURCE_ROOT = {SOURCE_ROOT}. "
        f"Expected directory: {os.path.join(SOURCE_ROOT, 'face2eeg')}"
    )

if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)


# ============================================================
# 4. Package imports
# ============================================================
from face2eeg.core.experiments import run_kfold_pipeline
from face2eeg.core.models      import try_gpu
from face2eeg.core.data        import clear_video_cache, clear_signals_cache

# Framework constants (from config)
from face2eeg.core.config import (
    SEED,
    FS_EEG,
    FS_VIDEO,
    VALID_SUBJECTS
)


# ============================================================
# 5. Reproducibility
# ============================================================
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# ============================================================
# Configuration
# ============================================================
BASE = os.path.join(DATA_ROOT, 'Experiments', 'Cross-Validation')

EXP_DIR     = os.path.join(BASE, 'Channel-Specificity')
RESULTS_DIR = os.path.join(EXP_DIR, 'Results')
MODELS_DIR  = os.path.join(EXP_DIR, 'Models')

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)


# ============================================================
# Experiment scope
# ============================================================
SUBJECTS = VALID_SUBJECTS
CHANNELS = ['Pz', 'Oz']      # Fz already analyzed separately


# ============================================================
# Hyperparameters
# ============================================================
TIME_WINDOW  = 2
FRAME_SIZE   = (32, 32)

EPOCHS       = 10
BATCH_SIZE   = 16
LR           = 1e-4
WEIGHT_DECAY = 1e-5
SCHED_STEP   = 5
SCHED_GAMMA  = 0.1

# Data generation
TRAIN_OVERLAP       = True
TRAIN_OVERLAP_RATIO = 0.5
TEST_OVERLAP        = False
TEST_OVERLAP_RATIO  = 0.0

# Unique exp_start per channel (avoids checkpoint collisions)
EXP_START_PER_CHANNEL = {'Pz': 300, 'Oz': 400}


# ============================================================
# Device
# ============================================================
device = try_gpu()


# ============================================================
# Sanity check
# ============================================================
print("EXP_DIR     :", EXP_DIR)
print("RESULTS_DIR :", RESULTS_DIR)
print("MODELS_DIR  :", MODELS_DIR)
print()
print("SUBJECTS    :", SUBJECTS)
print("CHANNELS    :", CHANNELS)
print("TIME_WINDOW :", TIME_WINDOW)
print("FRAME_SIZE  :", FRAME_SIZE)
print()
print("EPOCHS      :", EPOCHS)
print("BATCH_SIZE  :", BATCH_SIZE)
print("LR          :", LR)
print("WEIGHT_DECAY:", WEIGHT_DECAY)
print("SCHED_STEP  :", SCHED_STEP)
print("SCHED_GAMMA :", SCHED_GAMMA)
print()
print("TRAIN_OVERLAP       :", TRAIN_OVERLAP)
print("TRAIN_OVERLAP_RATIO :", TRAIN_OVERLAP_RATIO)
print("TEST_OVERLAP        :", TEST_OVERLAP)
print()
print("EXP_START_PER_CHANNEL:")
for ch in CHANNELS:
    print(f"  {ch}  →  exp_start = {EXP_START_PER_CHANNEL[ch]}")
print()
print("FS_EEG      :", FS_EEG, " (from config)")
print("FS_VIDEO    :", FS_VIDEO, " (from config)")
print("device      :", device)

## Run: subject × channel loop

Loops over the 7 subjects and 2 posterior channels. Each run:
- Uses the same 5-fold GroupKFold splits
- Trains a fresh PhysNet-3DCNN-ED with MSE loss
- Saves results under `Results/sub{XX}_ch{CH}.pkl`
- Skips automatically if the file already exists
- Frees RAM between subjects

In [ ]:
results_all = {}   #  {(subject_id, channel): results_dict}

for subject_id in SUBJECTS:
    print(f"\n{'#'*70}")
    print(f"#  SUBJECT {subject_id:02d}/{SUBJECTS[-1]:02d}")
    print(f"{'#'*70}")

    # Free RAM between subjects
    clear_video_cache()
    clear_signals_cache()

    for channel in CHANNELS:
        print(f"\n{'='*60}")
        print(f"  Subject {subject_id} | Channel {channel}")
        print(f"{'='*60}")

        # ---- skip if already done ----
        save_path_result = os.path.join(
            RESULTS_DIR, f"sub{subject_id:02d}_ch{channel}.pkl"
        )
        if os.path.isfile(save_path_result):
            print(f"  [skip] already exists: {save_path_result}")
            continue

        # ---- run CV ----
        try:
            results = run_kfold_pipeline(
                subject_id=subject_id,
                channel_name=channel,
                time_window=TIME_WINDOW,

                # CV splits
                n_points=40, n_splits=5,

                # scaler
                scaler_method='z-score',

                # data generation
                frame_size=FRAME_SIZE,
                video_fr=FS_VIDEO,
                signal_fs=FS_EEG,
                y_type='RawEEG',

                # train
                train_overlap=TRAIN_OVERLAP,
                train_overlap_ratio=TRAIN_OVERLAP_RATIO,
                train_shuffle_output=True,
                train_signal_normalize=True,

                # test
                test_overlap=TEST_OVERLAP,
                test_overlap_ratio=TEST_OVERLAP_RATIO,
                test_shuffle_output=False,
                test_signal_normalize=True,

                # model
                video_length=None,
                signal_length=None,

                # training
                epochs=EPOCHS,
                batch_size=BATCH_SIZE,
                learning_rate=LR,
                weight_decay=WEIGHT_DECAY,
                loss_fn=nn.MSELoss(),
                scheduler_step_size=SCHED_STEP,
                scheduler_gamma=SCHED_GAMMA,

                # prediction
                predict_batch_size=BATCH_SIZE,   

                # output
                model_exp_start=EXP_START_PER_CHANNEL[channel],
                save_dir=MODELS_DIR,
                device=device,
                return_history=True,
                verbose=True,
                root_dir=DATA_ROOT,
            )

        except Exception as e:
            print(f"  !! ERROR for subject={subject_id}, channel={channel}: {e}")
            traceback.print_exc()
            continue   # next channel/subject

        # ---- store ----
        results_all[(subject_id, channel)] = results

        # ---- save ----
        with open(save_path_result, 'wb') as f:
            pickle.dump(results, f)
        print(f"  saved: {save_path_result}")

        # ---- free GPU memory ----
        if device.type == 'cuda':
            torch.cuda.empty_cache()

In [ ]:
print("\n" + "="*70)
print("DONE — Channel Specificity")
print("="*70)
print(f"Results: {RESULTS_DIR}")
print(f"Models : {MODELS_DIR}")
print()

for subject_id in SUBJECTS:
    for channel in CHANNELS:
        p = os.path.join(RESULTS_DIR, f'sub{subject_id:02d}_ch{channel}.pkl')
        ok = "✅" if os.path.isfile(p) else "❌"
        print(f"  {ok}  S{subject_id:02d} | {channel}  {os.path.basename(p)}")